# Receipt Analytics Dashboard
A visual summary of your 6-month receipt data.  
Set `CSV_PATH` below to your file, or leave it as-is — sample data will be used if the file isn't found.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

# ── configure ──────────────────────────────────────────────────────────────
CSV_PATH = r'C:\Users\schraidera\Documents\Auto-OCR\receipts_count_6month.csv'
CSV_SEP  = ';'

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})
print('Libraries loaded.')

In [ ]:
# ── load data (falls back to sample if file not found) ─────────────────────
def make_sample_data():
    rng = np.random.default_rng(42)
    categories = ['Supermarket', 'Pharmacy', 'Restaurant', 'Gas Station',
                  'Electronics', 'Clothing', 'Bakery', 'Coffee Shop']
    stores = [
        'Whole Foods', 'CVS', 'Olive Garden', 'Shell', 'Best Buy',
        'H&M', 'Panera', 'Starbucks', 'Trader Joe\'s', 'Walgreens',
        'Applebee\'s', 'BP', 'Target', 'Rite Aid', 'McDonald\'s',
    ]
    cat_map = dict(zip(stores, [
        'Supermarket','Pharmacy','Restaurant','Gas Station','Electronics',
        'Clothing','Bakery','Coffee Shop','Supermarket','Pharmacy',
        'Restaurant','Gas Station','Supermarket','Pharmacy','Restaurant',
    ]))
    dates = pd.date_range('2024-01-01', periods=180, freq='D')
    rows = []
    for d in dates:
        n_stores = rng.integers(3, 8)
        chosen = rng.choice(stores, n_stores, replace=False)
        for s in chosen:
            rows.append({
                'date': d,
                'store': s,
                'category': cat_map[s],
                'n_receipts': int(rng.integers(1, 12)),
                'total_spend': round(float(rng.uniform(5, 200)), 2),
            })
    return pd.DataFrame(rows)

try:
    df = pd.read_csv(CSV_PATH, sep=CSV_SEP)
    # try to parse a date column if present
    date_cols = [c for c in df.columns if 'date' in c.lower() or 'data' in c.lower()]
    if date_cols:
        df[date_cols[0]] = pd.to_datetime(df[date_cols[0]], errors='coerce')
        df = df.rename(columns={date_cols[0]: 'date'})
    print(f'Loaded real data: {len(df):,} rows, {df.shape[1]} columns')
    USING_SAMPLE = False
except FileNotFoundError:
    df = make_sample_data()
    print('CSV not found — using sample data (180 days, 15 stores).')
    USING_SAMPLE = True

df.head(3)

In [ ]:
# ── overview ───────────────────────────────────────────────────────────────
total_receipts = df['n_receipts'].sum() if 'n_receipts' in df.columns else len(df)

if 'date' in df.columns:
    date_min = df['date'].min().strftime('%b %d, %Y')
    date_max = df['date'].max().strftime('%b %d, %Y')
    date_range_str = f'{date_min} → {date_max}'
else:
    date_range_str = 'N/A'

summary = pd.DataFrame({
    'Metric': ['Total rows', 'Total receipts', 'Date range', 'Columns', 'Missing values'],
    'Value': [
        f'{len(df):,}',
        f'{total_receipts:,}',
        date_range_str,
        ', '.join(df.columns.tolist()),
        str(df.isnull().sum().sum()),
    ]
})
summary.style.hide(axis='index').set_caption('Dataset Overview')

In [ ]:
# ── spend / receipts over time ─────────────────────────────────────────────
if 'date' in df.columns:
    count_col = 'n_receipts' if 'n_receipts' in df.columns else df.select_dtypes('number').columns[0]
    daily = df.groupby('date')[count_col].sum().reset_index()
    daily['rolling7'] = daily[count_col].rolling(7, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(daily['date'], daily[count_col], color='#a8d8ea', width=0.9, label='Daily')
    ax.plot(daily['date'], daily['rolling7'], color='#2c7bb6', lw=1.8, label='7-day avg')
    ax.set_title('Receipts Over Time', fontweight='bold')
    ax.set_ylabel('Receipts')
    ax.legend(frameon=False, fontsize=8)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print('No date column found — skipping time chart.')

In [ ]:
# ── category breakdown ─────────────────────────────────────────────────────
cat_col = 'category' if 'category' in df.columns else None
count_col = 'n_receipts' if 'n_receipts' in df.columns else df.select_dtypes('number').columns[0]

if cat_col:
    cat_data = (
        df.groupby(cat_col)[count_col]
        .sum()
        .sort_values()
    )
    colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(cat_data)))

    fig, ax = plt.subplots(figsize=(6, max(3, len(cat_data) * 0.45)))
    bars = ax.barh(cat_data.index, cat_data.values, color=colors)
    ax.bar_label(bars, fmt='{:,.0f}', padding=4, fontsize=8)
    ax.set_title('Receipts by Category', fontweight='bold')
    ax.set_xlabel('Total Receipts')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.show()
else:
    # fallback: show top stores/rows by numeric column
    top = df.nlargest(15, count_col)
    top_col = top.select_dtypes('object').iloc[:, 0] if not top.select_dtypes('object').empty else top.index
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(range(len(top)), top[count_col].values, color='#2c7bb6')
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top_col.values, fontsize=8)
    ax.set_title(f'Top 15 by {count_col}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── day-of-week heatmap ────────────────────────────────────────────────────
if 'date' in df.columns:
    count_col = 'n_receipts' if 'n_receipts' in df.columns else df.select_dtypes('number').columns[0]
    tmp = df.copy()
    tmp['week'] = tmp['date'].dt.isocalendar().week.astype(int)
    tmp['dow']  = tmp['date'].dt.day_name()
    order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    pivot = tmp.groupby(['week','dow'])[count_col].sum().unstack('dow').reindex(columns=order)

    fig, ax = plt.subplots(figsize=(7, max(3, len(pivot) * 0.22)))
    sns.heatmap(
        pivot, ax=ax,
        cmap='YlOrRd', linewidths=0.3, linecolor='white',
        annot=len(pivot) <= 15, fmt='.0f', annot_kws={'size': 7},
        cbar_kws={'label': 'Receipts', 'shrink': 0.6},
    )
    ax.set_title('Receipt Activity Heatmap (by Week × Day)', fontweight='bold')
    ax.set_ylabel('Week #')
    ax.set_xlabel('')
    plt.tight_layout()
    plt.show()
else:
    print('No date column — skipping heatmap.')

In [ ]:
# ── auto-generated plain-English summary ───────────────────────────────────
count_col = 'n_receipts' if 'n_receipts' in df.columns else df.select_dtypes('number').columns[0]
total = int(df[count_col].sum())

date_line = ''
busiest_day_line = ''
if 'date' in df.columns:
    d0 = df['date'].min().strftime('%B %d, %Y')
    d1 = df['date'].max().strftime('%B %d, %Y')
    date_line = f'spanning **{d0}** to **{d1}**'
    tmp = df.copy()
    tmp['dow'] = tmp['date'].dt.day_name()
    busiest = tmp.groupby('dow')[count_col].sum().idxmax()
    busiest_day_line = f'Your busiest day of the week is **{busiest}**.'

top_cat_line = ''
if 'category' in df.columns:
    top_cat = df.groupby('category')[count_col].sum().idxmax()
    top_cat_line = f'The top category is **{top_cat}**.'

source_note = '_\\* Sample data — connect your CSV to see real insights._' if USING_SAMPLE else ''

md = f"""
## Summary

You have **{total:,} receipts** {date_line}.  
{busiest_day_line}  
{top_cat_line}

{source_note}
"""
display(Markdown(md))